# 05 - Importación de variables ML

## Importar librerías

In [1]:
from pathlib import Path
from zipfile import ZipFile
from datetime import datetime, timezone
import hashlib
import json
import requests
import pandas as pd

## Mejorar la visualización

In [2]:
# Que muestre todas las columnas
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

## Establecer las rutas

In [3]:
RAIZ = Path.cwd().resolve()
for carpeta in [RAIZ, *RAIZ.parents]:
    if (carpeta / 'requirements.txt').is_file() and (carpeta / 'data').is_dir():
        RAIZ = carpeta
        break
else:
    raise FileNotFoundError('Abra el notebook desde el proyecto.')

RAW = RAIZ / 'data/raw'
PROCESSED = RAIZ / 'data/processed'
REPORTS = RAIZ / 'data/reports/05_importacion_ml'
ANIOS = list(range(2017, 2025))
archivos = {}

## Reutilizar o descargar los originales

In [4]:
def descargar(url, archivo, parametros = None):
    archivo.parent.mkdir(parents=True, exist_ok=True)
    if archivo.is_file():
        return archivo
    temporal = archivo.with_suffix(archivo.suffix + '.part')
    with requests.get(url, params = parametros, stream = True, timeout = (15, 90)) as respuesta:
        respuesta.raise_for_status()
        with temporal.open('wb') as salida:
            for bloque in respuesta.iter_content(1024 * 1024):
                salida.write(bloque)
        esperado = respuesta.headers.get('Content-Length')
        if esperado and not respuesta.headers.get('Content-Encoding'):
            assert temporal.stat().st_size == int(esperado), f'Descarga incompleta: {archivo.name}'
    assert temporal.stat().st_size > 0
    if archivo.suffix in ['.zip', '.xlsx']:
        with ZipFile(temporal) as paquete:
            assert paquete.testzip() is None, f'Archivo dañado: {archivo.name}'
    elif archivo.suffix == '.json':
        json.loads(temporal.read_text(encoding='utf-8-sig'))
    temporal.replace(archivo)
    return archivo

def registrar(archivo, grupo, fuente, url):
    with archivo.open('rb') as entrada:
        huella = hashlib.file_digest(entrada, 'sha256').hexdigest()
    archivos[str(archivo.relative_to(RAIZ))] = {
        'grupo': grupo, 'fuente': fuente, 'url': url,
        'bytes': archivo.stat().st_size, 'sha256': huella
    }

## Importar indicadores del Banco Mundial

In [5]:
def importar_indicador(codigo, fuente_id, grupo, fuente):
    archivo = RAW / 'variables_ml/world_bank' / f'{codigo}_2017_2024.json'
    url = f'https://api.worldbank.org/v2/country/all/indicator/{codigo}'
    parametros = {'format': 'json', 'source': fuente_id, 'date': '2017:2024', 'per_page': 20000}
    if not archivo.exists():
        registros, metadatos, pagina, paginas = [], [], 1, 1
        while pagina <= paginas:
            respuesta = requests.get(url, params = {**parametros, 'page': pagina}, timeout = (15, 60))
            respuesta.raise_for_status()
            meta, datos = respuesta.json()
            assert isinstance(datos, list), f'Respuesta vacía: {codigo}'
            registros.extend(datos)
            metadatos.append(meta)
            paginas = int(meta['pages'])
            pagina += 1
        assert len(registros) == int(metadatos[0]['total'])
        contenido = {'url': url, 'parametros': parametros, 'registros': registros,
            'metadatos_paginas': metadatos, 'fecha_descarga_utc': datetime.now(timezone.utc).isoformat()}
        archivo.parent.mkdir(parents=True, exist_ok=True)
        temporal = archivo.with_suffix('.json.part')
        temporal.write_text(json.dumps(contenido, ensure_ascii=False), encoding='utf-8')
        temporal.replace(archivo)
    contenido = json.loads(archivo.read_text(encoding='utf-8'))
    assert int(contenido['parametros']['source']) == fuente_id
    registros = [r for r in contenido['registros'] if r['indicator']['id'] == codigo]
    assert registros and {int(r['date']) for r in registros} == set(ANIOS)
    registrar(archivo, grupo, fuente, contenido['url'])
    return pd.json_normalize(registros)

## Variables externas

### Banco Mundial - Variables de destino

In [6]:
indicadores_wdi = [
    ('NY.GDP.MKTP.PP.KD', 'pib_ppa_constante_2021'),
    ('SP.POP.TOTL', 'poblacion_total'),
    ('NV.IND.TOTL.KD', 'va_industrial_constante_2015_usd'),
    ('BX.KLT.DINV.WD.GD.ZS', 'ied_entrada_pct_pib'),
    ('EG.ELC.ACCS.ZS', 'acceso_electricidad_pct')
]
datos_wdi = {}
for codigo, variable in indicadores_wdi:
    datos_wdi[variable] = importar_indicador(codigo, 2, 'Externa', 'Banco Mundial - WDI')
assert datos_wdi['pib_ppa_constante_2021']['indicator.value'].str.contains('constant 2021 international', regex=False).all()
assert datos_wdi['va_industrial_constante_2015_usd']['indicator.value'].str.contains('constant 2015 US$', regex=False).all()
pd.DataFrame([(variable, len(tabla)) for variable, tabla in datos_wdi.items()], columns=['Variable', 'Registros'])

,Variable,Registros
0,pib_ppa_constante_2021,2128
1,poblacion_total,2128
2,va_industrial_constante_2015_usd,2128
3,ied_entrada_pct_pib,2128
4,acceso_electricidad_pct,2128


### BEA y FRED - Contexto industrial de Estados Unidos

In [7]:
archivo_industria_usa = RAW / 'variables_ml/complementos/usa_va_industrial_2017_2024.csv'
industria_usa = pd.read_csv(archivo_industria_usa, float_precision = 'round_trip')
registrar(archivo_industria_usa, 'Externa', 'BEA/FRED - Proxy industrial USA (elaboración propia)',
    'https://fred.stlouisfed.org/series/RVAMA')
industria_usa

,iso3_d,year,va_industrial_constante_2015_usd
0,USA,2017,"3,467,730,441,672.64"
1,USA,2018,"3,597,342,058,113.87"
2,USA,2019,"3,665,576,154,691.27"
3,USA,2020,"3,577,309,704,827.69"
4,USA,2021,"3,741,760,139,083.35"
5,USA,2022,"3,638,400,063,352.46"
6,USA,2023,"3,793,098,953,688.47"
7,USA,2024,"3,940,512,331,475.78"


### WGI - Estabilidad política

In [8]:
estabilidad_politica = importar_indicador('GOV_WGI_PV.EST', 3, 'Externa', 'Banco Mundial - WGI')
estabilidad_politica.head()

,countryiso3code,date,value,unit,obs_status,decimal,indicator.id,indicator.value,country.id,country.value,scale
0,,2024,1.41,,,1,GOV_WGI_PV.EST,Political Stability - Governance estimate (app...,,Anguilla,NaN
1,,2023,1.41,,,1,GOV_WGI_PV.EST,Political Stability - Governance estimate (app...,,Anguilla,NaN
2,,2022,1.41,,,1,GOV_WGI_PV.EST,Political Stability - Governance estimate (app...,,Anguilla,NaN
3,,2021,1.40,,,1,GOV_WGI_PV.EST,Political Stability - Governance estimate (app...,,Anguilla,NaN
4,,2020,1.41,,,1,GOV_WGI_PV.EST,Political Stability - Governance estimate (app...,,Anguilla,NaN


### CEPII - BACI mundial

In [9]:
url_baci = 'https://www.cepii.fr/DATA_DOWNLOAD/baci/data/BACI_HS17_V202601.zip'
archivo_baci = descargar(url_baci, RAW / 'baci/BACI_HS17_V202601.zip')
with ZipFile(archivo_baci) as paquete:
    assert all(f'BACI_HS17_Y{anio}_V202601.csv' in paquete.namelist() for anio in ANIOS)
    with paquete.open('BACI_HS17_Y2017_V202601.csv') as entrada:
        baci_mundial_muestra = pd.read_csv(entrada, dtype = {'k': str}, nrows = 5)
registrar(archivo_baci, 'Externa', 'CEPII - BACI mundial', url_baci)
baci_mundial_muestra

,t,i,j,k,v,q
0,2017,4,12,130120,5.95,1.40
1,2017,4,12,130190,5.12,2.32
2,2017,4,12,401031,0.09,0.00
3,2017,4,12,853890,0.30,0.02
4,2017,4,36,071320,3.54,2.45


### SFC - Tasa representativa del mercado

In [10]:
url_trm = 'https://www.datos.gov.co/resource/32sa-8pi3.json'
parametros_trm = {'$where': "vigenciahasta >= '2017-01-01T00:00:00' AND vigenciadesde < '2025-01-01T00:00:00'",
    '$order': 'vigenciadesde', '$limit': 5000}
archivo_trm = descargar(url_trm, RAW / 'variables_ml/precios_trm/trm_sfc_2017_2024.json', parametros_trm)
registros_trm = json.loads(archivo_trm.read_text(encoding='utf-8'))
assert isinstance(registros_trm, list) and 0 < len(registros_trm) < parametros_trm['$limit']
trm = pd.DataFrame(registros_trm)
registrar(archivo_trm, 'Externa', 'SFC - TRM', url_trm)
trm.head()

,valor,unidad,vigenciadesde,vigenciahasta
0,3000.71,COP,2016-12-30T00:00:00.000,2017-01-03T00:00:00.000
1,2981.06,COP,2017-01-04T00:00:00.000,2017-01-04T00:00:00.000
2,2965.36,COP,2017-01-05T00:00:00.000,2017-01-05T00:00:00.000
3,2941.08,COP,2017-01-06T00:00:00.000,2017-01-06T00:00:00.000
4,2919.01,COP,2017-01-07T00:00:00.000,2017-01-10T00:00:00.000


### Banco Mundial - Precios de café y petróleo

In [11]:
url_precios = 'https://thedocs.worldbank.org/en/doc/74e8be41ceb20fa0da750cda2f6b9e4e-0050012026/related/CMO-Historical-Data-Annual.xlsx'
archivo_precios = descargar(url_precios, RAW / 'variables_ml/precios_trm/CMO-Historical-Data-Annual.xlsx')
precios_internacionales = pd.read_excel(archivo_precios, sheet_name = 'Annual Prices (Nominal)', header = 6,
    usecols = ['Unnamed: 0', 'Coffee, Arabica', 'Crude oil, Brent'])
registrar(archivo_precios, 'Externa', 'Banco Mundial - Precios internacionales', url_precios)
precios_internacionales.head()

,Unnamed: 0,"Crude oil, Brent","Coffee, Arabica"
0,NaN,($/bbl),($/kg)
1,"1,960.00",1.60,0.92
2,"1,961.00",1.60,0.90
3,"1,962.00",1.50,0.83
4,"1,963.00",1.50,0.81


## Variables internas

### CEPII - Exportaciones colombianas

In [12]:
archivo_colombia = PROCESSED / 'baci/baci_exports_COL_2017_2024_HS2017_V202601.parquet'
exportaciones_colombia = pd.read_parquet(archivo_colombia)
registrar(archivo_colombia, 'Interna', 'CEPII - BACI colombiano', url_baci)
exportaciones_colombia.head()

,year,reporter_code,reporter_iso3,partner_code,partner_iso3,partner_name,flow,frequency,hs6,hs6_name,hs4,hs2,hs_revision,trade_value_thousand_usd,trade_value_usd,quantity_tonnes,quantity_kg,source,source_version
0,2017,170,COL,8,ALB,Albania,X,A,080390,"Fruit, edible: bananas, other than plantains, ...",0803,08,HS2017,"1,676.91","1,676,907.00","1,933.51","1,933,513.00",BACI,202601
1,2017,170,COL,8,ALB,Albania,X,A,090111,Coffee: not roasted or decaffeinated,0901,09,HS2017,145.49,"145,493.00",40.21,"40,210.00",BACI,202601
2,2017,170,COL,8,ALB,Albania,X,A,170114,"Sugars: cane sugar, raw, in solid form, other ...",1701,17,HS2017,15.52,"15,523.00",25.00,"25,000.00",BACI,202601
3,2017,170,COL,8,ALB,Albania,X,A,250700,Kaolin and other kaolinic clays: whether or no...,2507,25,HS2017,50.00,"50,000.00",100.00,"100,000.00",BACI,202601
4,2017,170,COL,8,ALB,Albania,X,A,270400,"Coke and semi-coke: of coal, lignite or peat, ...",2704,27,HS2017,946.76,"946,757.00","5,917.23","5,917,230.00",BACI,202601


### DANE - Contexto productivo HS6

In [13]:
archivo_contexto_dane = RAW / 'variables_ml/complementos/contexto_dane_hs6_anio.parquet'
contexto_dane = pd.read_parquet(archivo_contexto_dane)
registrar(archivo_contexto_dane, 'Interna', 'DANE - Contexto HS6 (elaboración propia)',
    'https://www.dane.gov.co/files/operaciones/PIB/anex-CuentasNalANuales-OfertaUtilizacionPreciosCorrientes-2024p.xlsx')
contexto_dane.head()

,hs6,year,sector_cn,estado_concordancia,nombre_sector_cn,va_sector_col_miles_millones_cop,ci_sector_col_miles_millones_cop
0,010121,2017,A0102,unica,"Ganadería, caza y actividades de servicios con...","14,639.00","16,149.00"
1,010121,2018,A0102,unica,"Ganadería, caza y actividades de servicios con...","15,224.00","16,788.00"
2,010121,2019,A0102,unica,"Ganadería, caza y actividades de servicios con...","15,735.00","17,787.00"
3,010121,2020,A0102,unica,"Ganadería, caza y actividades de servicios con...","16,980.00","18,498.00"
4,010121,2021,A0102,unica,"Ganadería, caza y actividades de servicios con...","21,931.00","22,560.00"


### Registraduría - Calendario presidencial

In [14]:
elecciones_presidenciales = pd.DataFrame({'year': [2018, 2022]})
elecciones_presidenciales

,year
0,2018
1,2022


## Revisar las variables importadas

In [15]:
seleccion = pd.DataFrame([
    ('Externa', 'Banco Mundial - WDI', 'pib_ppa_constante_2021'),
    ('Externa', 'Banco Mundial - WDI', 'poblacion_total'),
    ('Externa', 'Banco Mundial - WDI y complemento BEA/FRED USA', 'va_industrial_constante_2015_usd'),
    ('Externa', 'Banco Mundial - WDI', 'ied_entrada_pct_pib'),
    ('Externa', 'Banco Mundial - WDI', 'acceso_electricidad_pct'),
    ('Externa', 'Banco Mundial - WGI', 'wgi_estabilidad_politica'),
    ('Externa', 'CEPII - BACI mundial', 'importaciones_producto_total_usd'),
    ('Externa', 'CEPII - BACI mundial', 'proveedores_ex_col'),
    ('Externa', 'SFC - TRM', 'trm_usd_por_cop'),
    ('Externa', 'Banco Mundial - Precios internacionales', 'precio_cafe_arabica_usd_kg'),
    ('Externa', 'Banco Mundial - Precios internacionales', 'precio_petroleo_brent_usd_barril'),
    ('Interna', 'CEPII - BACI colombiano', 'participacion_oferta_col'),
    ('Interna', 'DANE - Contexto HS6 (elaboración propia)', 'va_sector_col_miles_millones_cop'),
    ('Interna', 'DANE - Contexto HS6 (elaboración propia)', 'ci_sector_col_miles_millones_cop'),
    ('Interna', 'Registraduría', 'eleccion_presidencial_col'),
], columns = ['Grupo', 'Fuente', 'Variable'])
assert len(seleccion) == 15 and seleccion['Variable'].is_unique
seleccion

,Grupo,Fuente,Variable
0,Externa,Banco Mundial - WDI,pib_ppa_constante_2021
1,Externa,Banco Mundial - WDI,poblacion_total
2,Externa,Banco Mundial - WDI y complemento BEA/FRED USA,va_industrial_constante_2015_usd
3,Externa,Banco Mundial - WDI,ied_entrada_pct_pib
4,Externa,Banco Mundial - WDI,acceso_electricidad_pct
5,Externa,Banco Mundial - WGI,wgi_estabilidad_politica
6,Externa,CEPII - BACI mundial,importaciones_producto_total_usd
7,Externa,CEPII - BACI mundial,proveedores_ex_col
8,Externa,SFC - TRM,trm_usd_por_cop
9,Externa,Banco Mundial - Precios internacionales,precio_cafe_arabica_usd_kg


## Guardar el registro de importación

In [16]:
REPORTS.mkdir(parents=True, exist_ok=True)
registro = {'fecha_utc': datetime.now(timezone.utc).isoformat(), 'years': ANIOS,
    'variables': seleccion.to_dict('records'), 'archivos': archivos,
    'parametros_trm': parametros_trm, 'calendario_presidencial': elecciones_presidenciales.to_dict('records')}
(REPORTS / 'importacion.json').write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame.from_dict(archivos, orient = 'index')[['grupo', 'fuente', 'bytes']].rename_axis('Archivo')

,grupo,fuente,bytes
Archivo,,,
data\raw\variables_ml\world_bank\NY.GDP.MKTP.PP.KD_2017_2024.json,Externa,Banco Mundial - WDI,812634
data\raw\variables_ml\world_bank\SP.POP.TOTL_2017_2024.json,Externa,Banco Mundial - WDI,475222
data\raw\variables_ml\world_bank\NV.IND.TOTL.KD_2017_2024.json,Externa,Banco Mundial - WDI,860652
data\raw\variables_ml\world_bank\BX.KLT.DINV.WD.GD.ZS_2017_2024.json,Externa,Banco Mundial - WDI,581051
data\raw\variables_ml\world_bank\EG.ELC.ACCS.ZS_2017_2024.json,Externa,Banco Mundial - WDI,524353
data\raw\variables_ml\complementos\usa_va_industrial_2017_2024.csv,Externa,BEA/FRED - Proxy industrial USA (elaboración p...,274
data\raw\variables_ml\world_bank\GOV_WGI_PV.EST_2017_2024.json,Externa,Banco Mundial - WGI,469612
data\raw\baci\BACI_HS17_V202601.zip,Externa,CEPII - BACI mundial,794583540
data\raw\variables_ml\precios_trm\trm_sfc_2017_2024.json,Externa,SFC - TRM,293838
